# Second Ear: train the risk model

Fine-tunes DistilBERT on `BothBosu/scam-dialogue` so it can score a running call transcript from 0 to 1.
It trains on prefixes of each dialogue so the score rises early, and it reads the last 256 tokens, as the live scorer does.
It does not tune on the test split, touch any recorded call, or print a dialogue.

**Run it:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all. Approve the Google Drive prompt once.

**What it leaves in Drive, under `MyDrive/second-ear/`:**

- `risk-scorer.zip`: unzip into `<repo>/models/risk-scorer/` so that `models/risk-scorer/config.json` exists.
- `eval/results/model.json`: copy to `<repo>/eval/results/model.json`.
- `checkpoints/<RUN_NAME>/`: the newest finished epoch. After a disconnect, Run all resumes from it.

Training data is BothBosu only. This build has no matched-pairs set.

The order of the cells matters: tags and the leading greeting are stripped, the split is made by conversation, and only then are prefixes built. Last, every train and val example gets one copy with its first 5 to 15 words removed (seeded), so the model cannot rely on the templated opening. The test split gets no augmentation.

The leading greeting is removed from train, val and test alike, because in this dataset "hello" or "hi" alone nearly gives the label away. The headline test rows use that greeting-free text; a second pair of rows scores the original text so the difference is visible.

In [ ]:
# 1. Config: every knob in one place. Lines marked SPEC are frozen by docs/SPEC.md section 6.
DATASET_REPO = "BothBosu/scam-dialogue"
DATASET_REVISION = "321b961b5ae353e19ed479b960658dcd223d5e06"  # the revision checked on 2026-11-14
TRAIN_FILE = "scam-dialogue_train.csv"
TEST_FILE = "scam-dialogue_test.csv"
EXPECTED_COLUMNS = ["dialogue", "type", "label"]
EXPECTED_TRAIN_ROWS = 1280
EXPECTED_TEST_ROWS = 320

SEED = 42
VAL_FRACTION = 0.10
PREFIX_PERCENTS = (20, 40, 60, 80, 100)  # SPEC
EARLY_PERCENT = 40  # SPEC: the early-warning test
OPENING_CROP_WORDS = (5, 15)  # v3: each train/val example also appears with its first 5 to 15 words removed

BASE_MODEL = "distilbert-base-uncased"  # SPEC
MAX_LENGTH = 256  # SPEC
TRUNCATION_SIDE = "left"  # SPEC
EPOCHS = 3  # SPEC
LEARNING_RATE = 2e-5  # SPEC
USE_FP16 = True  # SPEC; takes effect on the GPU
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.06
MAX_GRAD_NORM = 1.0
THRESHOLD = 0.5  # cut on P(scam) for the reported metrics; never tuned

SERVER_TRANSFORMERS = "4.57.6"  # requirements.txt pin; the server loads the export with this version
REQUIRE_GPU = True  # set False only for a CPU dry run

RUN_NAME = "distilbert-prefix-v3-opencrop"  # change it to start a new run instead of resuming
DRIVE_MOUNT = "/content/drive"
DRIVE_ROOT = "/content/drive/MyDrive/second-ear"
WORK_DIR = "/content/second-ear"

In [ ]:
# 2. Dependencies: install only what this runtime lacks.
# Colab already has torch and transformers. The one thing it may lack is the exact
# transformers version the server loads the exported folder with.
import importlib
import importlib.metadata
import subprocess
import sys


def installed_version(package: str) -> str | None:
    """Version of an installed package, or None when it is absent."""
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None


found = installed_version("transformers")
if found == SERVER_TRANSFORMERS:
    print(f"transformers {found} matches the server. Nothing to install.")
elif "transformers" in sys.modules or "tokenizers" in sys.modules:
    raise RuntimeError(
        f"transformers {found} is already loaded and the server uses {SERVER_TRANSFORMERS}. "
        "Runtime > Restart session, then Runtime > Run all."
    )
else:
    print(f"transformers is {found}; installing {SERVER_TRANSFORMERS} to match the server. This takes a moment.")
    print("pip may warn that other preinstalled packages want newer versions. That is expected.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"transformers=={SERVER_TRANSFORMERS}"])
    importlib.invalidate_caches()
    # The install also swaps huggingface_hub. If the runtime had it loaded, drop it so the new one is imported.
    for name in [name for name in sys.modules if name.split(".")[0] == "huggingface_hub"]:
        del sys.modules[name]
    if installed_version("transformers") != SERVER_TRANSFORMERS:
        raise RuntimeError(f"pip finished but transformers is {installed_version('transformers')}.")
    print("transformers is now", installed_version("transformers"))

In [ ]:
# 3. GPU check and Drive mount, first, so the one Drive prompt comes at the start of Run all.
import os

import torch

if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU, then Runtime > Run all.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FP16 = USE_FP16 and DEVICE.type == "cuda"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None
print(f"device: {DEVICE} ({GPU_NAME or 'no GPU'}), fp16: {FP16}, torch {torch.__version__}")

from google.colab import drive

drive.mount(DRIVE_MOUNT)


def require_drive() -> None:
    """Stop before writing to a Drive path that is not mounted, for example after the final flush.

    Without this, a write after an unmount would quietly land on the runtime's own disk.
    """
    if not os.path.isdir(os.path.dirname(DRIVE_ROOT)):
        raise RuntimeError(
            f"{os.path.dirname(DRIVE_ROOT)} is not there, so Google Drive is not mounted. "
            "Re-run the '3. GPU check and Drive mount' cell first."
        )


require_drive()
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints", RUN_NAME)
DRIVE_RESULTS_DIR = os.path.join(DRIVE_ROOT, "eval", "results")
EXPORT_DIR = os.path.join(WORK_DIR, "risk-scorer")
for folder in (CHECKPOINT_DIR, DRIVE_RESULTS_DIR, WORK_DIR):
    os.makedirs(folder, exist_ok=True)
print("checkpoints go to", CHECKPOINT_DIR)

In [ ]:
# Shared helpers. This cell is identical in train_risk_model.ipynb and baselines.ipynb.
import csv
import json
import os
import re
from collections import Counter
from datetime import datetime, timezone

from huggingface_hub import hf_hub_download

TAG_PATTERN = re.compile(r"\b(?:caller|receiver)\s*:\s*", re.IGNORECASE)
GREETING_PATTERN = re.compile(
    r"^\s*(?:hello|hi|hey|good\s+(?:morning|afternoon|evening))\b[\s,.!?;:\-\u2013\u2014]*", re.IGNORECASE
)
RESULTS_SCHEMA = "second-ear/results-v1"
METRIC_NAMES = ("precision", "recall", "f1")


def load_split(repo: str, revision: str, filename: str, expected_columns: list[str]) -> list[dict]:
    """Download one CSV at a pinned revision and return its rows, labels as ints."""
    path = hf_hub_download(repo, filename, repo_type="dataset", revision=revision)
    with open(path, newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames != expected_columns:
            raise RuntimeError(f"{filename} has columns {reader.fieldnames}, expected {expected_columns}")
        return [{**row, "label": int(row["label"])} for row in reader]


def describe_split(name: str, rows: list[dict]) -> None:
    """Print columns, row count and label balance. Never prints a dialogue."""
    labels = Counter(row["label"] for row in rows)
    kinds = Counter((row["type"], row["label"]) for row in rows)
    print(f"{name}: {len(rows)} rows, columns {list(rows[0])}")
    print(f"  label balance: scam (1) = {labels[1]}, legitimate (0) = {labels[0]}")
    for (kind, label), count in sorted(kinds.items()):
        print(f"  type {kind:<14} label {label}: {count}")


def strip_tags(dialogue: str) -> str:
    """Remove the inline speaker tags and collapse whitespace. Live transcripts have none."""
    return " ".join(TAG_PATTERN.sub(" ", dialogue).split())


def strip_greeting(text: str) -> str:
    """Remove one leading greeting (hello, hi, hey, good morning / afternoon / evening) and the
    punctuation after it. The same rule for every row and both labels: in this dataset the
    greeting alone nearly gives the label away."""
    return GREETING_PATTERN.sub("", text, count=1)


def prepared(rows: list[dict]) -> list[dict]:
    """New rows: `text` has tags and the leading greeting removed; `original` keeps the greeting."""
    out = []
    for row in rows:
        original = strip_tags(row["dialogue"])
        out.append({"text": strip_greeting(original), "original": original, "type": row["type"], "label": row["label"]})
    return out


def greeting_counts(rows: list[dict]) -> dict:
    """How many prepared rows lost a greeting, in all and per label."""
    changed = Counter(row["label"] for row in rows if row["text"] != row["original"])
    return {"rows": len(rows), "greeting_removed": sum(changed.values()), "scam": changed[1], "legitimate": changed[0]}


def prefix_at(text: str, percent: int) -> str:
    """The first `percent` % of the text's words, rounded up, never fewer than one word."""
    words = text.split()
    keep = max(1, (len(words) * percent + 99) // 100)
    return " ".join(words[:keep])


def binary_metrics(labels: list[int], predictions: list[int]) -> dict:
    """Precision, recall and F1 for the scam class, with the counts behind them.

    A metric with a zero denominator is None, not 0, so no table can show a made-up number.
    """
    pairs = list(zip(labels, predictions))
    tp = sum(1 for label, guess in pairs if label == 1 and guess == 1)
    fp = sum(1 for label, guess in pairs if label == 0 and guess == 1)
    fn = sum(1 for label, guess in pairs if label == 1 and guess == 0)
    tn = sum(1 for label, guess in pairs if label == 0 and guess == 0)
    return {
        "n": len(pairs),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": tp / (tp + fp) if tp + fp else None,
        "recall": tp / (tp + fn) if tp + fn else None,
        "f1": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else None,
    }


def correct_by_type(rows: list[dict], predictions: list[int]) -> dict:
    """Per scenario type: how many dialogues there are and how many were labelled correctly."""
    totals = Counter(row["type"] for row in rows)
    correct = Counter(row["type"] for row, guess in zip(rows, predictions) if guess == row["label"])
    return {kind: {"n": totals[kind], "correct": correct[kind]} for kind in sorted(totals)}


def rounded(value: float | None) -> float | None:
    """Four decimals for the results file; None stays None."""
    return None if value is None else round(value, 4)


def show(value: float | None) -> str:
    """A metric as text; an undefined one reads n/a."""
    return "n/a" if value is None else f"{value:.4f}"


def metric_rows(test: str, metrics: dict, context: str) -> list[dict]:
    """Table-ready rows for one test set: test, metric, value, n, notes (docs/SPEC.md section 8).

    n is each metric's own denominator: dialogues flagged as scam for precision, scam
    dialogues for recall, every dialogue scored for F1.
    """
    counts = f"tp={metrics['tp']} fp={metrics['fp']} fn={metrics['fn']} tn={metrics['tn']}"
    denominators = {
        "precision": (metrics["tp"] + metrics["fp"], "dialogues flagged as scam"),
        "recall": (metrics["tp"] + metrics["fn"], "scam dialogues"),
        "f1": (metrics["n"], "dialogues scored"),
    }
    rows = []
    for name in METRIC_NAMES:
        n, meaning = denominators[name]
        notes = f"n = {meaning}; {metrics['n']} scored in all; {counts}; {context}"
        rows.append({"test": test, "metric": name, "value": rounded(metrics[name]), "n": n, "notes": notes})
    return rows


def write_results(payload: dict, path: str) -> str:
    """Write one results file as indented JSON and return its path."""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)
        handle.write("\n")
    return path


def utc_now() -> str:
    """Current UTC time, to the second, for the results file."""
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

In [ ]:
# 4. Load both CSVs straight from the Hugging Face hub. Counts only; no dialogue is printed.
train_raw = load_split(DATASET_REPO, DATASET_REVISION, TRAIN_FILE, EXPECTED_COLUMNS)
test_raw = load_split(DATASET_REPO, DATASET_REVISION, TEST_FILE, EXPECTED_COLUMNS)

for name, rows, expected in (("train", train_raw, EXPECTED_TRAIN_ROWS), ("test", test_raw, EXPECTED_TEST_ROWS)):
    describe_split(name, rows)
    if len(rows) != expected:
        raise RuntimeError(f"{name} has {len(rows)} rows, expected {expected}. Re-check docs/SPEC.md section 6.")

In [ ]:
# 5. Strip the inline "caller:" / "receiver:" tags, then the leading greeting. Train, val and test alike.
# Live transcripts carry no speaker labels. The greeting goes because here it nearly gives the label away.
import statistics

train_all = prepared(train_raw)
test_rows = prepared(test_raw)
GREETING_REMOVED = {"train_csv": greeting_counts(train_all), "test_csv": greeting_counts(test_rows)}

tagged_before = sum(1 for row in train_raw + test_raw if TAG_PATTERN.search(row["dialogue"]))
tagged_after = sum(1 for row in train_all + test_rows if TAG_PATTERN.search(row["text"]))
still_greeting = sum(1 for row in train_all + test_rows if GREETING_PATTERN.match(row["text"]))
word_counts = [len(row["text"].split()) for row in train_all + test_rows]
print(f"rows carrying a speaker tag: {tagged_before} before, {tagged_after} after")
print(f"leading greeting removed: {GREETING_REMOVED}")
print(f"rows that still open with a greeting: {still_greeting}")
print(f"words per dialogue: min {min(word_counts)}, median {statistics.median(word_counts):.0f}, max {max(word_counts)}")
if tagged_after:
    raise RuntimeError("Some dialogues still carry a speaker tag.")
if any(not row["text"] for row in train_all + test_rows):
    raise RuntimeError("A dialogue is empty after stripping.")

In [ ]:
# 6. Split by conversation FIRST, then augment train and val. The given test CSV is the test set.
import random


def remove_leaks(rows: list[dict], held_out: set[str]) -> tuple[list[dict], dict]:
    """Drop training conversations that repeat, or that appear word for word in the test set."""
    kept, seen, dropped = [], set(), {"duplicate_within_train": 0, "also_in_test": 0}
    for row in rows:
        if row["text"] in held_out:
            dropped = {**dropped, "also_in_test": dropped["also_in_test"] + 1}
        elif row["text"] in seen:
            dropped = {**dropped, "duplicate_within_train": dropped["duplicate_within_train"] + 1}
        else:
            seen.add(row["text"])
            kept.append(row)
    return kept, dropped


def split_by_conversation(rows: list[dict], val_fraction: float, seed: int) -> tuple[list[dict], list[dict]]:
    """Stratified by type. Every conversation lands in exactly one of train and val."""
    rng = random.Random(seed)
    by_type: dict[str, list[int]] = {}
    for index, row in enumerate(rows):
        by_type.setdefault(row["type"], []).append(index)
    val_ids: set[int] = set()
    for kind in sorted(by_type):
        ids = list(by_type[kind])
        rng.shuffle(ids)
        val_ids.update(ids[: round(len(ids) * val_fraction)])
    train = [row for index, row in enumerate(rows) if index not in val_ids]
    val = [row for index, row in enumerate(rows) if index in val_ids]
    return train, val


def augment(rows: list[dict], percents: tuple[int, ...]) -> list[dict]:
    """One example per prefix length, same label. A prefix that repeats within a dialogue is kept once."""
    examples = []
    for row in rows:
        seen = set()
        for percent in percents:
            text = prefix_at(row["text"], percent)
            if text not in seen:
                seen.add(text)
                examples.append({"text": text, "label": row["label"], "percent": percent})
    return examples


def opening_crops(examples: list[dict], low: int, high: int, seed: int) -> tuple[list[dict], int]:
    """For every example, a copy with its first k words removed, k uniform in low..high, same label.

    The opening of these dialogues is templated and nearly gives the label away; the copies
    make the model work without it. An example of k words or fewer would leave nothing, so
    it gets no copy. The second value returned counts those.
    """
    rng = random.Random(seed)
    copies, skipped = [], 0
    for example in examples:
        words = example["text"].split()
        k = rng.randint(low, high)
        if len(words) <= k:
            skipped += 1
        else:
            copies.append({"text": " ".join(words[k:]), "label": example["label"],
                           "percent": example["percent"], "cropped_words": k})
    return copies, skipped


def opening_key(text: str, words: int = 12) -> str:
    """Lowercased, punctuation-free first words: the fingerprint of a templated opening."""
    return " ".join(re.sub(r"[^a-z0-9 ]", "", text.lower()).split()[:words])


def given_away_by_opening(train: list[dict], test: list[dict], field: str, words: int) -> int:
    """How many test labels the opening alone gets right, with no model.

    Each test dialogue's first words are looked up among the training conversations and
    given the label most of those carry. `field` picks the text with or without the greeting.
    """
    votes: dict[str, Counter] = {}
    for row in train:
        votes.setdefault(opening_key(row[field], words), Counter())[row["label"]] += 1
    majority = {key: counter.most_common(1)[0][0] for key, counter in votes.items()}
    return sum(1 for row in test if majority.get(opening_key(row[field], words)) == row["label"])


test_texts = {row["text"] for row in test_rows}
train_clean, DROPPED = remove_leaks(train_all, test_texts)
train_convs, val_convs = split_by_conversation(train_clean, VAL_FRACTION, SEED)

train_conv_texts = {row["text"] for row in train_convs}
val_conv_texts = {row["text"] for row in val_convs}
if train_conv_texts & val_conv_texts or train_conv_texts & test_texts or val_conv_texts & test_texts:
    raise RuntimeError("A conversation sits in two splits.")
if len(test_rows) != len(test_raw):
    raise RuntimeError("The test split changed size.")

# Augmentation happens only now, and only for train and val: prefixes first, then one
# opening-cropped copy of every prefix example. The test split gets neither.
train_prefixes = augment(train_convs, PREFIX_PERCENTS)
val_prefixes = augment(val_convs, PREFIX_PERCENTS)
train_crops, train_crops_skipped = opening_crops(train_prefixes, *OPENING_CROP_WORDS, SEED + 1)
val_crops, val_crops_skipped = opening_crops(val_prefixes, *OPENING_CROP_WORDS, SEED + 2)
train_examples = train_prefixes + train_crops
val_examples = val_prefixes + val_crops
OPENING_CROP = {
    "words_removed": f"uniform {OPENING_CROP_WORDS[0]} to {OPENING_CROP_WORDS[1]}, seeded",
    "train_prefix_examples": len(train_prefixes),
    "train_cropped_copies": len(train_crops),
    "train_too_short_for_a_copy": train_crops_skipped,
    "val_prefix_examples": len(val_prefixes),
    "val_cropped_copies": len(val_crops),
    "val_too_short_for_a_copy": val_crops_skipped,
    "fewest_words_left_in_a_copy": min(len(example["text"].split()) for example in train_crops + val_crops),
}
early_test_texts = [prefix_at(row["text"], EARLY_PERCENT) for row in test_rows]

# How independent is the test split really? Counted, never assumed.
# The last two counts use test labels, for this diagnostic only. Nothing here feeds training.
OPENING_WORDS = 6
train_example_texts = {example["text"] for example in train_examples}
train_openings = {opening_key(row["text"]) for row in train_convs}
OVERLAP = {
    "test_dialogues": len(test_rows),
    "sharing_first_12_words_with_a_training_dialogue": sum(
        1 for row in test_rows if opening_key(row["text"]) in train_openings
    ),
    "full_identical_to_a_training_example": sum(1 for row in test_rows if row["text"] in train_example_texts),
    "early_prefix_identical_to_a_training_example": sum(1 for text in early_test_texts if text in train_example_texts),
    "label_given_away_by_first_6_words": given_away_by_opening(train_convs, test_rows, "text", OPENING_WORDS),
    "label_given_away_by_first_6_words_with_greeting": given_away_by_opening(
        train_convs, test_rows, "original", OPENING_WORDS
    ),
}

print(f"dropped from train: {DROPPED}")
print(f"conversations: train {len(train_convs)}, val {len(val_convs)}, test {len(test_rows)}")
print(f"opening-crop copies: {OPENING_CROP}")
print(f"examples in all: train {len(train_examples)}, val {len(val_examples)}")
print(f"val label balance: {dict(sorted(Counter(row['label'] for row in val_convs).items()))}")
print(f"test overlap with training text: {OVERLAP}")

In [ ]:
# 7. Tokenizer: last 256 tokens, cut from the left, exactly what the live scorer reads.
from torch.utils.data import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, truncation_side=TRUNCATION_SIDE)
if tokenizer.truncation_side != TRUNCATION_SIDE:
    raise RuntimeError("The tokenizer is not truncating from the left.")

probe_ids = tokenizer("start " + "filler " * 1000 + "finish", truncation=True, max_length=MAX_LENGTH)["input_ids"]
probe_text = tokenizer.decode(probe_ids, skip_special_tokens=True)
if len(probe_ids) != MAX_LENGTH or not probe_text.endswith("finish") or "start" in probe_text:
    raise RuntimeError("Left truncation self-check failed: the tail of a long text was not kept.")
print(f"left truncation holds: {len(probe_ids)} tokens kept, the end of the text survives")


class EncodedDataset(Dataset):
    """Token ids and labels for a list of examples. Padding happens per batch."""

    def __init__(self, examples: list[dict]) -> None:
        encoded = tokenizer([example["text"] for example in examples], truncation=True, max_length=MAX_LENGTH)
        self.input_ids = encoded["input_ids"]
        self.attention_mask = encoded["attention_mask"]
        self.labels = [example["label"] for example in examples]

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int) -> dict:
        return {
            "input_ids": self.input_ids[index],
            "attention_mask": self.attention_mask[index],
            "labels": self.labels[index],
        }


def score_texts(model, tokenizer, texts: list[str]) -> list[float]:
    """P(scam) for each text, in fp32: softmax over the logits, index 1. The server does the same."""
    model.eval()
    scores: list[float] = []
    with torch.no_grad():
        for start in range(0, len(texts), EVAL_BATCH_SIZE):
            batch = tokenizer(
                texts[start : start + EVAL_BATCH_SIZE],
                truncation=True,
                max_length=MAX_LENGTH,
                padding=True,
                return_tensors="pt",
            ).to(model.device)
            scores.extend(torch.softmax(model(**batch).logits.float(), dim=-1)[:, 1].cpu().tolist())
    return scores


train_set = EncodedDataset(train_examples)
at_limit = sum(1 for ids in train_set.input_ids if len(ids) == MAX_LENGTH)
print(f"training examples: {len(train_set)}, of which {at_limit} fill all {MAX_LENGTH} tokens")

In [ ]:
# 8. Train: plain PyTorch loop, fp16 on the GPU, one checkpoint to Drive per finished epoch.
import hashlib
import math
import time

from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding, get_linear_schedule_with_warmup

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2, id2label={0: "LEGIT", 1: "SCAM"}, label2id={"LEGIT": 0, "SCAM": 1}
).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
steps_per_epoch = math.ceil(len(train_set) / BATCH_SIZE)
total_steps = steps_per_epoch * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP_FRACTION), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=FP16)
collator = DataCollatorWithPadding(tokenizer)

# A checkpoint is only resumed when it was made by this exact setup.
RUN_FINGERPRINT = hashlib.sha256(
    json.dumps(
        [DATASET_REVISION, SEED, VAL_FRACTION, PREFIX_PERCENTS, BASE_MODEL, MAX_LENGTH, TRUNCATION_SIDE, EPOCHS,
         LEARNING_RATE, BATCH_SIZE, WEIGHT_DECAY, WARMUP_FRACTION, FP16, len(train_set)]
    ).encode()
).hexdigest()[:16]
# ... and on these exact training examples and the remaining loop settings.
DATA_FINGERPRINT = hashlib.sha256(
    json.dumps([MAX_GRAD_NORM, [[example["text"], example["label"]] for example in train_examples]]).encode()
).hexdigest()[:16]


def checkpoint_path(epoch: int) -> str:
    return os.path.join(CHECKPOINT_DIR, f"checkpoint-epoch{epoch}.pt")


def save_checkpoint(epoch: int, train_seconds: float, history: list[dict]) -> None:
    """Write the full training state. The previous epoch's file is kept as a spare.

    Drive uploads in the background, so the newest file may not be safe yet when the
    runtime dies. Keeping one older checkpoint means a disconnect still costs one epoch.
    """
    require_drive()
    state = {
        "fingerprint": RUN_FINGERPRINT,
        "data_fingerprint": DATA_FINGERPRINT,
        "epoch": epoch,
        "train_seconds": train_seconds,
        "history": history,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
    }
    torch.save(state, checkpoint_path(epoch))
    for older in range(1, epoch - 1):
        if os.path.exists(checkpoint_path(older)):
            os.remove(checkpoint_path(older))


def load_latest_checkpoint() -> dict | None:
    """The newest readable checkpoint, or None. A half-written file is skipped, not fatal."""
    for epoch in range(EPOCHS, 0, -1):
        if not os.path.exists(checkpoint_path(epoch)):
            continue
        try:
            state = torch.load(checkpoint_path(epoch), map_location=DEVICE, weights_only=True)
        except Exception as error:
            print(f"checkpoint for epoch {epoch} is unreadable ({type(error).__name__}); trying an older one")
            continue
        if state["fingerprint"] != RUN_FINGERPRINT or state.get("data_fingerprint") != DATA_FINGERPRINT:
            raise RuntimeError(
                "The checkpoint in Drive comes from a different config, dataset or training text. "
                "Change RUN_NAME in the config cell to start a new run."
            )
        return state
    return None


def train_one_epoch(epoch: int) -> float:
    """One pass over the training examples. Returns the mean training loss."""
    loader = DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collator,
        generator=torch.Generator().manual_seed(SEED + epoch),
    )
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(loader, start=1):
        batch = {key: value.to(DEVICE) for key, value in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=FP16):
            loss = model(**batch).loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
        if step % 30 == 0 or step == len(loader):
            print(f"  epoch {epoch} step {step}/{len(loader)} loss {total_loss / step:.4f}")
    return total_loss / len(loader)


state = load_latest_checkpoint()
RESUMED = state is not None
start_epoch, train_seconds, history = 0, 0.0, []
if state is not None:
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    scaler.load_state_dict(state["scaler"])
    start_epoch, train_seconds, history = state["epoch"], state["train_seconds"], list(state["history"])
    print(f"resuming after epoch {start_epoch} ({train_seconds:.0f} s of training already done)")
del state
EPOCHS_TRAINED_NOW = EPOCHS - start_epoch
if EPOCHS_TRAINED_NOW == 0:
    print(f"NOTHING IS TRAINED IN THIS SESSION: all {EPOCHS} epochs are already in the Drive checkpoint.")
    print("The weights evaluated and exported below are that checkpoint's. Change RUN_NAME to train again.")

val_texts = [example["text"] for example in val_examples]
val_labels = [example["label"] for example in val_examples]

# The clock covers training and validation. Checkpoint writes to Drive are left out, so the
# number is the same whether or not the run was interrupted.
for epoch in range(start_epoch + 1, EPOCHS + 1):
    clock = time.perf_counter()
    train_loss = train_one_epoch(epoch)
    val = binary_metrics(val_labels, [int(score >= THRESHOLD) for score in score_texts(model, tokenizer, val_texts)])
    train_seconds += time.perf_counter() - clock
    history = history + [
        {"epoch": epoch, "train_loss": round(train_loss, 4), "val_n": val["n"],
         "val_precision": rounded(val["precision"]), "val_recall": rounded(val["recall"]), "val_f1": rounded(val["f1"])}
    ]
    print(f"epoch {epoch}/{EPOCHS}: train loss {train_loss:.4f}, val precision {show(val['precision'])}, "
          f"recall {show(val['recall'])}, F1 {show(val['f1'])} (n={val['n']}), {train_seconds:.0f} s so far")
    save_checkpoint(epoch, train_seconds, history)
    print(f"  checkpoint saved: {checkpoint_path(epoch)}")

print(f"training time: {train_seconds:.1f} s for {EPOCHS} epochs" + (" (resumed run)" if RESUMED else ""))

In [ ]:
# 9. Evaluate once on the held-out test split: full dialogues, and their first 40 % of words.
# Headline: the greeting removed, as in training. Second: the original text with its greeting,
# which the model never trained on, so the before and after sit side by side.
FULL_TEST = "bothbosu_test_full"
EARLY_TEST = f"bothbosu_test_prefix{EARLY_PERCENT}"
FULL_TEST_GREETING = f"{FULL_TEST}_with_greeting"
EARLY_TEST_GREETING = f"{EARLY_TEST}_with_greeting"
HEADLINE_TESTS = (FULL_TEST, EARLY_TEST)
test_labels = [row["label"] for row in test_rows]
test_sets = {
    FULL_TEST: [row["text"] for row in test_rows],
    EARLY_TEST: early_test_texts,
    FULL_TEST_GREETING: [row["original"] for row in test_rows],
    EARLY_TEST_GREETING: [prefix_at(row["original"], EARLY_PERCENT) for row in test_rows],
}

RESULTS = {}
for name, texts in test_sets.items():
    predictions = [int(score >= THRESHOLD) for score in score_texts(model, tokenizer, texts)]
    metrics = binary_metrics(test_labels, predictions)
    RESULTS[name] = {"metrics": metrics, "by_type": correct_by_type(test_rows, predictions)}
    print(f"{name}: precision {show(metrics['precision'])}, recall {show(metrics['recall'])}, "
          f"F1 {show(metrics['f1'])}, n={metrics['n']} "
          f"(tp={metrics['tp']} fp={metrics['fp']} fn={metrics['fn']} tn={metrics['tn']})")

# Diagnostic, not a headline number: the same two tests with the templated opening cut off.
# If these fall well below the rows above, the model is leaning on how the call starts.
OPENING_CUT = 12


def without_opening(text: str) -> str:
    """The text minus its first words; unchanged when it is no longer than the cut."""
    rest = text.split()[OPENING_CUT:]
    return " ".join(rest) if rest else text


OPENING_REMOVED = {}
for name in HEADLINE_TESTS:
    scores = score_texts(model, tokenizer, [without_opening(text) for text in test_sets[name]])
    metrics = binary_metrics(test_labels, [int(score >= THRESHOLD) for score in scores])
    OPENING_REMOVED[name] = {key: rounded(value) if key in METRIC_NAMES else value for key, value in metrics.items()}
    print(f"{name}, first {OPENING_CUT} words removed: precision {show(metrics['precision'])}, "
          f"recall {show(metrics['recall'])}, F1 {show(metrics['f1'])}, n={metrics['n']}")

In [ ]:
# 10. Export model + tokenizer, prove the folder loads the way the server loads it, zip it to Drive.
import shutil
import zipfile


def file_sha256(path: str) -> str:
    """sha256 of a file, read in blocks."""
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        block = handle.read(1 << 20)
        while block:
            digest.update(block)
            block = handle.read(1 << 20)
    return digest.hexdigest()


if os.path.isdir(EXPORT_DIR):
    shutil.rmtree(EXPORT_DIR)
model.save_pretrained(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)

reloaded_tokenizer = AutoTokenizer.from_pretrained(EXPORT_DIR)
reloaded_model = AutoModelForSequenceClassification.from_pretrained(EXPORT_DIR).to("cpu")
if reloaded_tokenizer.truncation_side != TRUNCATION_SIDE:
    raise RuntimeError("The exported tokenizer lost left truncation.")
if reloaded_model.config.id2label != {0: "LEGIT", 1: "SCAM"}:
    raise RuntimeError("The exported config lost its label names.")

sample = test_sets[FULL_TEST][:8] + test_sets[EARLY_TEST][:8]
drift = max(
    abs(a - b)
    for a, b in zip(score_texts(model, tokenizer, sample), score_texts(reloaded_model, reloaded_tokenizer, sample))
)
if drift > 1e-3:
    raise RuntimeError(f"The reloaded model scores differently (max difference {drift:.5f}).")
print(f"export reloads on CPU and scores the same (max difference {drift:.6f})")

# Six hand-written lines, not dataset rows. Informational only. Each message is scored bare
# and behind both greetings: training text has no leading greeting, live transcripts will.
SCAM_LINE = ("this is the fraud department your account is frozen and you must buy gift cards "
             "right now and read me the codes or you will be arrested")
EVERYDAY_LINE = "this is the pharmacy calling your prescription is ready you can pick it up any time today"
PROBES = {
    "scam-like, no greeting": SCAM_LINE,
    "scam-like, opens with hello": f"hello {SCAM_LINE}",
    "scam-like, opens with hi": f"hi {SCAM_LINE}",
    "everyday, no greeting": EVERYDAY_LINE,
    "everyday, opens with hello": f"hello {EVERYDAY_LINE}",
    "everyday, opens with hi": f"hi {EVERYDAY_LINE}",
}
PROBE_SCORES = {}
for name, score in zip(PROBES, score_texts(reloaded_model, reloaded_tokenizer, list(PROBES.values()))):
    PROBE_SCORES[name] = round(score, 3)
    print(f"  {name}: {score:.3f}")

exported = sorted(os.listdir(EXPORT_DIR))
if "config.json" not in exported:
    raise RuntimeError("config.json is missing from the export.")
MODEL_SHA256 = file_sha256(os.path.join(EXPORT_DIR, "model.safetensors"))
require_drive()
zip_local = os.path.join(WORK_DIR, "risk-scorer.zip")
with zipfile.ZipFile(zip_local, "w", zipfile.ZIP_DEFLATED) as archive:
    for name in exported:
        archive.write(os.path.join(EXPORT_DIR, name), arcname=name)  # flat: config.json at the zip root
ZIP_DRIVE = os.path.join(DRIVE_ROOT, "risk-scorer.zip")
shutil.copyfile(zip_local, ZIP_DRIVE)
print(f"zipped {exported}")
print(f"wrote {ZIP_DRIVE} ({os.path.getsize(ZIP_DRIVE) / 1e6:.0f} MB)")

In [ ]:
# 11. Write model.json: one table-ready row per metric, each with its n. Runs after the export is proven.
import transformers

context = f"scam is the positive class; P(scam) >= {THRESHOLD}; {MAX_LENGTH} tokens, left-truncated"
early = f"first {EARLY_PERCENT} % of words"
removed = "leading greeting removed, as in training"
kept = "original text with its greeting, which training never saw"
rows = metric_rows(FULL_TEST, RESULTS[FULL_TEST]["metrics"], f"{removed}; {context}")
rows = rows + metric_rows(EARLY_TEST, RESULTS[EARLY_TEST]["metrics"], f"{early}; {removed}; {context}")
rows = rows + metric_rows(FULL_TEST_GREETING, RESULTS[FULL_TEST_GREETING]["metrics"], f"{kept}; {context}")
rows = rows + metric_rows(EARLY_TEST_GREETING, RESULTS[EARLY_TEST_GREETING]["metrics"], f"{early}; {kept}; {context}")
rows = rows + [
    {
        "test": "training",
        "metric": "train_seconds",
        "value": round(train_seconds, 1),
        "n": len(train_examples),
        "notes": f"{GPU_NAME or 'CPU'}, fp16={FP16}, {EPOCHS} epochs; training and validation, checkpoint writes "
                 f"excluded; n = training examples; resumed={RESUMED}",
    }
]

payload = {
    "schema": RESULTS_SCHEMA,
    "system": "distilbert",
    "recipe": "v3: prefixes, leading greeting removed, plus an opening-cropped copy of every train and val example",
    "run_name": RUN_NAME,
    "created_utc": utc_now(),
    "dataset": {
        "repo": DATASET_REPO,
        "revision": DATASET_REVISION,
        "train_rows": len(train_raw),
        "test_rows": len(test_raw),
    },
    "preprocessing": {
        "speaker_tags": "removed",
        "leading_greeting": "removed from train, val and test: hello, hi, hey, good morning / afternoon / "
                            "evening, plus the punctuation after it; one rule for every row and both labels",
        "greeting_removed": GREETING_REMOVED,
        "headline_tests": list(HEADLINE_TESTS),
    },
    "split": {
        "seed": SEED,
        "val_fraction": VAL_FRACTION,
        "dropped_from_train": DROPPED,
        "train_conversations": len(train_convs),
        "val_conversations": len(val_convs),
        "test_conversations": len(test_rows),
        "prefix_percents": list(PREFIX_PERCENTS),
        "opening_crop": OPENING_CROP,
        "train_examples": len(train_examples),
        "val_examples": len(val_examples),
    },
    "training": {
        "base_model": BASE_MODEL,
        "max_length": MAX_LENGTH,
        "truncation_side": TRUNCATION_SIDE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "fp16": FP16,
        "device": DEVICE.type,
        "gpu": GPU_NAME,
        "train_seconds": round(train_seconds, 1),
        "resumed": RESUMED,
        "epochs_trained_this_session": EPOCHS_TRAINED_NOW,
        "val_by_epoch": history,
        "transformers": transformers.__version__,
        "torch": torch.__version__,
    },
    "threshold": THRESHOLD,
    "rows": rows,
    "confusion": {name: {key: result["metrics"][key] for key in ("tp", "fp", "fn", "tn")} for name, result in RESULTS.items()},
    "by_type": {name: result["by_type"] for name, result in RESULTS.items()},
    "test_overlap_with_training": OVERLAP,
    "diagnostics": {
        "opening_removed": {"words_removed": OPENING_CUT, **OPENING_REMOVED},
        "greeting_probe": {"note": "six hand-written lines, not dataset rows; P(scam) for each", **PROBE_SCORES},
    },
    "export": {"archive": "risk-scorer.zip", "files": exported, "model_safetensors_sha256": MODEL_SHA256},
    "caveats": [
        "Synthetic, templated data: "
        f"{OVERLAP['sharing_first_12_words_with_a_training_dialogue']} of {OVERLAP['test_dialogues']} test dialogues "
        "share their first 12 words with a training dialogue.",
        "The opening still gives the label away after the greeting is removed: looking up a test dialogue's "
        "first 6 words among the training conversations and taking their majority label is right for "
        f"{OVERLAP['label_given_away_by_first_6_words']} of {OVERLAP['test_dialogues']} test dialogues with no "
        f"model at all. With the greeting left in it was {OVERLAP['label_given_away_by_first_6_words_with_greeting']}.",
        f"{OVERLAP['early_prefix_identical_to_a_training_example']} of {OVERLAP['test_dialogues']} "
        f"{EARLY_PERCENT} % test prefixes are word for word identical to a training example.",
        "Trained on BothBosu only; no matched-pairs set.",
    ],
}

require_drive()
LOCAL_JSON = write_results(payload, os.path.join(WORK_DIR, "eval", "results", "model.json"))
drive_json = write_results(payload, os.path.join(DRIVE_RESULTS_DIR, "model.json"))
print(json.dumps(payload, indent=2))
print("\nwrote", drive_json)

In [ ]:
# 12. Finish: where the two files are, where they go, then flush Drive.
print("Done. In Google Drive:")
print("  MyDrive/second-ear/risk-scorer.zip")
print("  MyDrive/second-ear/eval/results/model.json")
print()
print("On the laptop:")
print("  1. Unzip risk-scorer.zip INTO <repo>/models/risk-scorer/ so that")
print("     <repo>/models/risk-scorer/config.json exists. The zip is flat: config.json is at its root.")
print("  2. Copy model.json to <repo>/eval/results/model.json")
print("  3. python -m server.preflight --skip-whisper   (it should report the scorer folder)")
print()
print("The score is softmax(logits)[1]: index 1 is SCAM, index 0 is LEGIT.")

try:
    drive.flush_and_unmount()
    print("Drive is flushed. Safe to close this tab.")
except Exception as error:
    print(f"Could not flush Drive ({type(error).__name__}). Keep the tab open until both files show in drive.google.com.")

# Last, so nothing above waits on the browser: offer model.json as a download too.
try:
    from google.colab import files

    files.download(LOCAL_JSON)
except Exception as error:
    print(f"No browser download ({type(error).__name__}). Use the copy in Drive.")